In [ ]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

# Moore's Shortest Path Algorithm

The function `shortestPath` takes two arguments.
- `source` is the start node.  
- `Edges` is a dictionary that encodes the set of edges of the graph.  For every node `x` the value of `Edges[x]` has the form
   $$ \bigl[ (y_1, l_1), \cdots, (y_n, l_n) \bigr]. $$
   This list is interpreted as follows: For every $i = 1,\cdots,n$ there is an edge
   $(x, y_i)$ pointing from $x$ to $y_i$ and this edge has the length $l_i$.
   
The task of the function `shortestPath(source, Edges)` is to compute the distances of all nodes in the graph from the node `source`.
This function works as follows:
- The variable `Distance` is a *dictionary*.  After the computation is
  finished, for every node $x$ such that $x$ is reachable from the node `source`,
  the value `Distance[x]` records the length of the shortest path from `source` to $x$.

  The node `source` has distance $0$ from the node `source` and initially this is
  all we know.  Hence, the dictionary `Distance` is initialized as `{ source: 0 }`.
- The variable `Fringe` is a list of those nodes that already have an estimate of
  their distance from the  node `source`.  Furthermore, if a node is in `Fringe`, then this
  node has at least one neighboring node whose distance from `source` hasn't been computed. 
  Initially we only know that the node `source` is connected to the node `source`.
  Therefore, we initialize the list `Fringe` as the list `[source]`. 
- Every iteration of the `while`-loop takes the first node `u` from the list `Fringe`.
- Next, we compute all nodes `v` that can be reached from `u` by an edge `(u, v)`. For every such node 
  `v` we check whether reaching `v` from `u` results in a shorter path than previously known.
  There are two cases.
  - If there is an edge from node `u` to a node `v` and  `Distance[v]` is still
    undefined, then we hadn't yet found a path leading to `v`.
  - Furthermore, there are those nodes `v` where we had already found a path leading from
    `source` to `v` but the length of this path is longer than the length of the path
    that we get when we first visit `u` and then proceed to `v` via the edge `(u,v)`.
    
  We compute the distance of the path leading from `source` to `u` and then to
  `v` in both of these cases and add `v` to the `Fringe`.
- The algorithm terminates when the list `Fringe` is empty because in that case we don't
  have any means left to improve our estimate of the distance function.

In [ ]:
function shortestPath(source: string, Edges: Map<string, [string, number][]>): Map<string, number> {
    const Distance = new Map<string, number>();
    Distance.set(source, 0);
    const Fringe: string[] = [source];
    while (Fringe.length > 0) {
        const u = Fringe.pop()!;
        const neighbors = Edges.get(u) || [];
        
        for (const [v, l] of neighbors) {
            const dv = Distance.get(v);
            if (dv === undefined || Distance.get(u)! + l < dv) {
                Distance.set(v, Distance.get(u)! + l);
                if (!Fringe.includes(v)) {
                    Fringe.push(v);
                }
            }
        }
    }
    return Distance;
}

## Code to Display the Directed Graph

The function $\texttt{toDot}(\texttt{source}, \texttt{Edges}, \texttt{Fringe}, \texttt{Distance})$ takes a graph that is represented by 
its `Edges`, a set of nodes `Fringe`, and a dictionary `Distance` that has the distance of a node from the node `source`.

In [ ]:
import { Graphviz } from "@hpcc-js/wasm";
import { display } from "tslab";

async function toDot(
    source: string, p: string | null, 
    Edges: Map<string, [string, number][]>, 
    Fringe: string[], 
    Distance: Map<string, number>
): Promise<void> {
    const V = new Set<string>();
    for (const x of Edges.keys()) {
        V.add(x);
    }
    let dot = 'digraph {\n';
    dot += 'rankdir=LR;\n';
    dot += 'size="9";\n';
    dot += 'node [shape=record style=rounded];\n\n';
    for (const x of V) {
        if (x === source) {
            if (x !== p && Fringe.includes(x)) {
                dot += `${x} [color=red shape=doublecircle];\n`;
            } else if (x === p) {
                dot += `${x} [color=magenta shape=doublecircle];\n`;
            } else {
                dot += `${x} [color=blue shape=doublecircle];\n`;
            }
        } else {
            const d = Distance.has(x) ? Distance.get(x)!.toString() : '';
            if (x === p) {
                dot += `${x} [label="{${x}|${d}}" color=magenta];\n`;
            } else if (Fringe.includes(x)) {
                dot += `${x} [label="{${x}|${d}}" color=red];\n`;
            } else {
                dot += `${x} [label="{${x}|${d}}"];\n`;
            }
        }
    }
    dot += '\n';
    for (const u of V) {
        const neighbors = Edges.get(u) || [];
        for (const [v, l] of neighbors) {
            dot += `${u} -> ${v} [label="${l}"];\n`;
        }
    }
    dot += '}\n';
    const gv = await Graphviz.load();
    const svg = gv.layout(dot, "svg", "dot");
    display.html(svg);
}

The version of `shortestPath` that is given below adds some animation to the algorithm.

In [ ]:
import { Graphviz } from "@hpcc-js/wasm";
import * as tslab from "tslab";

async function shortestPath(source: string, Edges: Map<string, [string, number][]>): Promise<Map<string, number>> {
    const Distance = new Map<string, number>();
    Distance.set(source, 0);
    const Fringe: string[] = [source];
    while (Fringe.length > 0) {
        await toDot(source, null, Edges, Fringe, Distance);
        console.log('_'.repeat(80));
        const u = Fringe.pop()!;
        await toDot(source, u, Edges, Fringe, Distance);
        console.log('_'.repeat(80));
        const neighbors = Edges.get(u) || [];
        for (const [v, l] of neighbors) {
            const dv = Distance.get(v);
            if (dv === undefined || Distance.get(u)! + l < dv) {
                Distance.set(v, Distance.get(u)! + l);
                if (!Fringe.includes(v)) {
                    Fringe.push(v);
                }
            }
        }
    }
    await toDot(source, null, Edges, Fringe, Distance);
    return Distance;
}

## Code for Testing

In [ ]:
const Edges = new Map<string, [string, number][]>([
  ['a', [['c', 2], ['b', 9]]],
  ['b', [['d', 1]]],
  ['c', [['e', 5], ['g', 3]]],
  ['d', [['f', 2], ['e', 4]]],
  ['e', [['f', 1], ['b', 2]]],
  ['f', [['h', 5]]],
  ['g', [['e', 1]]],
  ['h', []]
]);

In the diagrams below, the node `u` that is taken from the `Fringe` is colored in <em style="color:magenta; background-color:white">magenta</em>, 
while the nodes in the `Fringe` are colored in <em style="color:red; background-color:white">red</em>.  The start node is colored 
<em style="color:blue; background-color:white">blue</em> unless it is part of the `Fringe`or equal to the node `u` that is taken from the `Fringe`.

In [ ]:
const s = 'a';
const sp = await shortestPath(s, Edges);
for (const [node, dist] of sp.entries()) {
  console.log(`${node}: ${dist}`);
}